In [1]:
# Install required packages in a Jupyter notebook
%pip install psycopg2-binary
%pip install curl_cffi
%pip install undetected-chromedriver

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [7]:
import time
import random
from pathlib import Path
import pandas as pd
import undetected_chromedriver as uc

def extract_records(payload):
    """Safely unpacks API records across different response formats."""
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        if 'data' in payload:
            if isinstance(payload['data'], list):
                return payload['data']
            elif isinstance(payload['data'], dict) and 'data' in payload['data']:
                return payload['data']['data']
        if 'projects' in payload and isinstance(payload['projects'], list):
            return payload['projects']
    return []

def run_resilient_pipeline():
    # Changed path to the Downloads folder and updated the target filename
    downloads_dir = Path.home() / "Downloads"
    output_csv = downloads_dir / "dpwh_csv_finale_scraped.csv"
    watermark_file = downloads_dir / "dpwh_csv_watermark.txt"
    
    # 1,000 items per batch = ~266 total requests for 265,582 rows
    page_limit = 1000
    
    current_page = 1
    if watermark_file.exists():
        try:
            current_page = int(watermark_file.read_text().strip())
        except Exception:
            current_page = 1

    # Load master columns if resuming existing CSV to prevent column sliding
    master_columns = None
    if output_csv.exists() and output_csv.stat().st_size > 0:
        try:
            master_columns = pd.read_csv(output_csv, nrows=0, encoding="utf-8-sig").columns.tolist()
        except Exception:
            master_columns = None

    print(f"Launching authenticated browser context starting from Page {current_page}...")
    options = uc.ChromeOptions()
    options.add_argument("--start-maximized")
    
    driver = uc.Chrome(options=options)
    driver.set_script_timeout(45)

    try:
        # Step 1: Open root domain to clear Turnstile and establish session
        driver.get("https://transparency.dpwh.gov.ph/")
        print("Waiting 12 seconds for Cloudflare clearance and hydration...")
        time.sleep(12)

        while True:
            # Step 2: Inject fetch with cache-busting timestamp (_ts)
            cache_buster = int(time.time() * 1000)
            fetch_script = f"""
            var callback = arguments[0];
            fetch('https://api.transparency.dpwh.gov.ph/projects?page={current_page}&limit={page_limit}&_ts={cache_buster}', {{
                headers: {{
                    'Accept': 'application/json, text/plain, */*',
                    'Cache-Control': 'no-cache',
                    'Pragma': 'no-cache'
                }},
                credentials: 'include'
            }})
            .then(res => {{
                if (res.status !== 200) {{
                    callback({{status: res.status, data: null}});
                    return;
                }}
                return res.json();
            }})
            .then(data => {{
                if (data) callback({{status: 200, data: data}});
            }})
            .catch(err => callback({{status: 500, error: err.toString()}}));
            """

            try:
                result = driver.execute_async_script(fetch_script)
            except Exception as e:
                print(f"[{time.strftime('%H:%M:%S')}] Script communication hiccup: {e}. Retrying in 5s...")
                time.sleep(5)
                continue

            status = result.get("status") if result else 500

            # Step 3: Handle WAF re-challenges
            if status in [403, 429]:
                print(f"[{time.strftime('%H:%M:%S')}] Cloudflare challenge encountered (HTTP {status}).")
                print("Refreshing tab to obtain fresh Turnstile clearance...")
                driver.refresh()
                time.sleep(15)
                continue
            elif status != 200:
                print(f"[{time.strftime('%H:%M:%S')}] HTTP {status}. Waiting 10s before retry...")
                time.sleep(10)
                continue

            payload = result.get("data", {})
            items = extract_records(payload)

            if not items:
                print("\nReached the end of available records.")
                if watermark_file.exists():
                    watermark_file.unlink()
                break

            valid_items = [item for item in items if isinstance(item, dict)]

            if valid_items:
                batch_df = pd.json_normalize(valid_items)

                # Initialize master schema on the very first batch
                if master_columns is None:
                    master_columns = batch_df.columns.tolist()
                    batch_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
                else:
                    # Enforce exact column alignment across all batches
                    batch_df = batch_df.reindex(columns=master_columns)
                    batch_df.to_csv(output_csv, mode="a", header=False, index=False, encoding="utf-8-sig")

                # Verify different records are being pulled by sampling the first/last item
                first_id = valid_items[0].get("contract_id", valid_items[0].get("id", "N/A"))
                last_id = valid_items[-1].get("contract_id", valid_items[-1].get("id", "N/A"))
                print(f"[{time.strftime('%H:%M:%S')}] Page {current_page} synced: {len(valid_items):,} rows (ID {first_id} -> {last_id})")

            # Update checkpoint
            watermark_file.write_text(str(current_page + 1))

            if len(items) < page_limit:
                print("\nSuccessfully reached the final page.")
                if watermark_file.exists():
                    watermark_file.unlink()
                break

            current_page += 1
            # Throttled delay between batches
            time.sleep(random.uniform(3.0, 4.5))

    finally:
        driver.quit()

if __name__ == "__main__":
    run_resilient_pipeline()

Launching authenticated browser context starting from Page 1...
Waiting 12 seconds for Cloudflare clearance and hydration...
[03:06:27] Page 1 synced: 1,000 rows (ID N/A -> N/A)
[03:06:32] Page 2 synced: 1,000 rows (ID N/A -> N/A)
[03:06:44] Page 3 synced: 1,000 rows (ID N/A -> N/A)
[03:06:55] Page 4 synced: 1,000 rows (ID N/A -> N/A)
[03:07:07] Page 5 synced: 1,000 rows (ID N/A -> N/A)
[03:07:18] Page 6 synced: 1,000 rows (ID N/A -> N/A)
[03:07:30] Page 7 synced: 1,000 rows (ID N/A -> N/A)
[03:07:42] Page 8 synced: 1,000 rows (ID N/A -> N/A)
[03:07:54] Page 9 synced: 1,000 rows (ID N/A -> N/A)
[03:08:06] Page 10 synced: 1,000 rows (ID N/A -> N/A)
[03:08:18] Page 11 synced: 1,000 rows (ID N/A -> N/A)
[03:08:31] Page 12 synced: 1,000 rows (ID N/A -> N/A)
[03:08:43] Page 13 synced: 1,000 rows (ID N/A -> N/A)
[03:08:56] Page 14 synced: 1,000 rows (ID N/A -> N/A)
[03:09:08] Page 15 synced: 1,000 rows (ID N/A -> N/A)
[03:09:20] Page 16 synced: 1,000 rows (ID N/A -> N/A)
[03:09:33] Page 17 s